In [1]:
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from peft import LoraConfig, get_peft_model
import math
import os
import gc

from GPT_Config import GPTConfig
from GPT import GPT
from train_config import TrainerConfig
from train import LLMTrainer
from data_loader import LLMDataset
from builtTokenizer import TokenizerProcessor

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")

if device.type == 'mps':
    gc.collect()
    torch.mps.empty_cache()
    torch.backends.mps.enable_mps_fallback = True
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
    torch.mps.synchronize()
    print("MPS memory limit increased, cpu fallback initialised, cache cleaned")

Using MPS (Apple Silicon GPU)
Device: mps
MPS memory limit increased, cpu fallback initialised, cache cleaned


In [3]:
config = GPTConfig()
model = GPT(config, use_checkpointing=True)
#state_dict = torch.load("...path/model.pth", map_location=device)
#model.load_state_dict(state_dict)
model.to(device)

# For fine tuning using lora
# model.config.get = lambda key, default=None: getattr(model.config, key, default)

# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=["in_proj", "out_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type=None,
#     modules_to_save=["tok_emb"]
# )

# peft_model = get_peft_model(model, lora_config)
# peft_model.print_trainable_parameters()

GPT model initialized with 151.92M parameters


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (h): ModuleList(
      (0-11): 12 x AttentionBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): MultiSelfAttentionHead(
          (in_proj): Linear(in_features=768, out_features=2304, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): FeedForward(
          (w1_3): Linear(in_features=768, out_features=6144, bias=False)
          (w2): Linear(in_features=3072, out_features=768, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
RAW_DATA_DIR = "...path/dir_of_txts"
TOKENIZER_DIR = "...path/tokenizer.json"
CHECKPOINT_DIR = ""

NUM_EPOCHS = 20000
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 750
EVAL_INTERVAL = 40
LOG_INTERVAL = 10
CHECKPOINT_INTERVAL = 30
STEP_ACCUM = 32
BATCH_SIZE = 2

BLOCK_SIZE = config.block_size
N_EMBD = config.n_embd
N_HEAD = config.n_head
N_LAYER = config.n_layer
DROPOUT = config.dropout

In [5]:
tokenizer_processor = TokenizerProcessor(TOKENIZER_DIR)

In [6]:
train_ds = LLMDataset(RAW_DATA_DIR, tokenizer_processor, BLOCK_SIZE, split="train", target_mb=10)
val_ds = LLMDataset(RAW_DATA_DIR, tokenizer_processor, BLOCK_SIZE, split="validation")

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

Loading random 10MB subset (1 files) for training...
Loading all 1 files for validation...


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (NUM_EPOCHS - WARMUP_STEPS)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

In [ ]:
t_config = TrainerConfig(
    device=device, 
    block_size=BLOCK_SIZE,
    max_steps=NUM_EPOCHS,
    step_accum=STEP_ACCUM,
    log_interval=LOG_INTERVAL,
    eval_interval=EVAL_INTERVAL,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    checkpoint_dir=str(CHECKPOINT_DIR)
)

trainer = LLMTrainer(model, optimizer, scheduler, train_loader, val_loader, t_config)
trainer.train()